# 03 · Tumor segmentation (3D SegResNet) and localization

The 3D segmentation model looks at the whole MRI volume (4 modalities) and marks **every voxel** as tumor or not, for the
three BraTS regions: **WT** whole tumor, **TC** tumor core, **ET** enhancing tumor.
The default model is **SegResNet**, a residual 3D U-Net-style encoder–decoder (winner of the BraTS 2018 challenge).
From the predicted mask, plain code computes the **localization**: position, size in mm, volume, hemisphere.

**Runtime:** GPU (Runtime → Change runtime type → **T4 GPU**). **Needs:** notebook 01 finished (same `QUICK_RUN` setting).
The full training takes about 10–12 hours; the state is saved after every epoch, so a disconnect only loses the current
epoch: reopen, *Run all*, and it resumes. **Check the first epoch finishes** (no "CUDA out of memory").

The settings below are the final run. `QUICK_RUN = True` trains 2 short epochs on the quick data instead (smoke test).

Everything is saved to `runs/segmentation/<RUN_NAME>/` on Drive.

In [ ]:
# ======================= SETTINGS - edit these ===========================
PROJECT_DIR = '/content/drive/MyDrive/BrainTumor'
WORK_DIR    = '/content/work'
QUICK_RUN   = False            # False = real training. True = 2-epoch smoke test on the quick data

RUN_NAME    = 'segresnet_v1'   # same name = resume, new name = new run
MODEL_NAME  = 'segresnet'      # segresnet | unet3d | unet3d_large | dynunet

EPOCHS        = 100
BATCH_SIZE    = 2                 # if "CUDA out of memory": 1 (and a new RUN_NAME)
ROI           = (128, 128, 128)   # training patch size (voxels)
FG_PROB       = 0.33              # share of patches centred on tumor
LR            = 5e-4              # SegResNet; the plain U-Net (unet3d) used 1e-3
WEIGHT_DECAY  = 1e-5
WARMUP_EPOCHS = 2
VAL_EVERY     = 2                 # validate every N epochs ...
VAL_CASES_DURING_TRAINING = 40    # ... on this many validation patients (all of them at the end)
EARLY_STOP_PATIENCE = None        # in validations without improvement (None = train all epochs)

SW_BATCH      = 4                 # sliding-window batch at prediction time
SW_OVERLAP    = 0.5
THRESHOLD     = 0.5
ET_MIN_VOXELS = 0                 # optional: remove tiny ET predictions (BraTS trick, e.g. 500). 0 = off

AMP           = True
NUM_WORKERS   = None
SEED          = 42
KEEP_EVERY_EPOCH   = True         # keep the weights of every epoch (~20 MB each)
SAVE_PROBABILITIES = True         # save predicted probability maps of every val/test patient

In [ ]:
# ---- Setup: mount Google Drive, install missing packages, import the project code ----
import os
import sys

# Only used when testing outside Colab; on Colab these three lines change nothing.
PROJECT_DIR = os.environ.get('BT_PROJECT_DIR', PROJECT_DIR)
WORK_DIR = os.environ.get('BT_WORK_DIR', WORK_DIR)
QUICK_RUN = os.environ.get('BT_QUICK_RUN', '1' if QUICK_RUN else '0') == '1'

if PROJECT_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

CODE_DIR = os.path.join(PROJECT_DIR, 'code')
if not os.path.isfile(os.path.join(CODE_DIR, 'brats', '__init__.py')):
    raise FileNotFoundError(f'The "brats" folder was not found in {CODE_DIR}.\n'
                            'Upload everything from the "Code" folder to that Drive folder first (see README).')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from brats.notebook import install_missing
install_missing({'nibabel': 'nibabel', 'monai': 'monai', 'sklearn': 'scikit-learn'})

import copy
import time
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import monai
from IPython.display import display

from brats import config, dataset, env, evaluation, fileio, plots, segmentation, training

if QUICK_RUN:
    EPOCHS, VAL_EVERY, VAL_CASES_DURING_TRAINING, WARMUP_EPOCHS = 2, 1, 3, 0
paths = env.ProjectPaths(PROJECT_DIR, WORK_DIR, quick=QUICK_RUN).make()
env.print_header('QUICK RUN - smoke test only, the numbers mean nothing' if QUICK_RUN
                 else f'TRAINING RUN "{RUN_NAME}"')
device = env.get_device()
env.require_gpu(device, QUICK_RUN)
NUM_WORKERS = NUM_WORKERS if NUM_WORKERS is not None else env.default_num_workers()
env.set_seed(SEED)
print(f'device={device}  workers={NUM_WORKERS}  torch={torch.__version__}  monai={monai.__version__}')

## 1 · Copy the preprocessed data to the local disk

In [ ]:
cases_dir = dataset.stage_to_local_disk(paths)
cases, slices, split = dataset.load_metadata(paths)
case_ids = dataset.split_case_ids(split)
dataset.verify_staged(cases_dir, case_ids['train'] + case_ids['val'] + case_ids['test'])
rng = np.random.default_rng(SEED)
val_subset = sorted(rng.choice(case_ids['val'], size=min(VAL_CASES_DURING_TRAINING, len(case_ids['val'])),
                               replace=False))
print({name: len(ids) for name, ids in case_ids.items()}, f'| validation during training on {len(val_subset)} patients')

## 2 · Run folder, settings and environment (saved to Drive)

In [ ]:
run_dir = paths.run_dir('segmentation', RUN_NAME)
config_record = dict(
    task='segmentation', run_name=RUN_NAME, quick_run=QUICK_RUN, model_name=MODEL_NAME, epochs=EPOCHS,
    batch_size=BATCH_SIZE, roi=list(ROI), fg_prob=FG_PROB, lr=LR, weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS, val_every=VAL_EVERY, val_cases_during_training=len(val_subset),
    early_stop_patience=EARLY_STOP_PATIENCE, sw_batch=SW_BATCH, sw_overlap=SW_OVERLAP, threshold=THRESHOLD,
    et_min_voxels=ET_MIN_VOXELS, amp=AMP, seed=SEED, in_channels=list(config.MODALITIES),
    out_channels=list(config.REGIONS), norm_clip=config.NORM_CLIP,
    n_cases={name: len(ids) for name, ids in case_ids.items()}, data_dir=str(paths.data))
config_path = run_dir / 'config.json'
if config_path.is_file():
    previous = fileio.load_json(config_path)
    if previous.get('model_name') != MODEL_NAME or list(previous.get('roi', ROI)) != list(ROI):
        raise RuntimeError(f'Run "{RUN_NAME}" used {previous.get("model_name")} / ROI {previous.get("roi")}. '
                           'Use a new RUN_NAME.')
    fileio.save_json(config_record, run_dir / 'logs' / f'config_resume_{time.strftime("%Y%m%d_%H%M%S")}.json')
else:
    fileio.save_json(config_record, config_path)
env.save_environment(run_dir / 'logs')
env.snapshot_code(run_dir, paths.code, '03_segmentation_localization.ipynb')
fileio.save_csv(pd.DataFrame({'case_id': val_subset}), run_dir / 'logs' / 'val_subset_during_training.csv')
print('Run folder:', run_dir)

## 3 · Data loaders and a look at training patches

In [ ]:
train_set = segmentation.SegPatchDataset(case_ids['train'], cases_dir, ROI, FG_PROB, augment=True)
train_loader = training.make_loader(train_set, BATCH_SIZE, True, NUM_WORKERS, device,
                                    drop_last=len(train_set) >= 2 * BATCH_SIZE)
val_loader = training.make_loader(segmentation.SegCaseDataset(val_subset, cases_dir), 1, False, NUM_WORKERS, device,
                                  segmentation.single_item_collate)
print(f'{len(train_loader)} training iterations per epoch')

patches, targets = next(iter(train_loader))
print('patch batch:', tuple(patches.shape), patches.dtype, '| targets:', tuple(targets.shape), targets.dtype,
      f'| tumor voxels per patch: {targets[:, 0].flatten(1).sum(1).tolist()}')
assert patches.shape[1:] == (4,) + tuple(ROI) and targets.shape[1:] == (3,) + tuple(ROI)
assert torch.isfinite(patches.float()).all()
images, outlines, titles = [], [], []
for i in range(min(4, len(patches))):
    k = int(np.argmax(targets[i, 0].sum((0, 1)).numpy())) if targets[i, 0].any() else ROI[2] // 2
    for c, modality in enumerate(config.MODALITIES):
        images.append(plots.view2d(patches[i, c, :, :, k].float().numpy()))
        outlines.append([(plots.view2d(targets[i, r, :, :, k].numpy()), plots.REGION_COLORS[region])
                         for r, region in enumerate(config.REGIONS)])
        titles.append(f'patch {i} {plots.MOD_TITLES[modality]}')
fig = plots.image_grid(images, titles, contours=outlines, ncols=4, cell=2.2,
                       suptitle='Training patches (outlines: blue WT, orange TC, green ET)')
plots.save_fig(fig, run_dir / 'figures', '01_training_patches')
del patches, targets

## 4 · Model, optimizer, and resume from the last checkpoint (if any)

In [ ]:
model = segmentation.build_segmentation_model(MODEL_NAME).to(device)
n_params = training.count_parameters(model)
loss_fn = segmentation.BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = max(1, len(train_loader))
scheduler = training.make_scheduler(optimizer, EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch)
scaler = training.make_grad_scaler(device, AMP)
print(f'{MODEL_NAME}: {n_params / 1e6:.2f} M parameters')

run = training.ResumableTraining(run_dir, model, optimizer, scheduler, scaler, device, model_name=MODEL_NAME,
                                 config=config_record, metric_name='val_mean_dice', epochs=EPOCHS,
                                 patience=EARLY_STOP_PATIENCE, keep_every_epoch=KEEP_EVERY_EPOCH)

## 5 · Training
Every epoch: `last.pt` (full state) and `history.csv` are saved. Every `VAL_EVERY` epochs the model is validated
(sliding window over whole volumes) and `best.pt` is updated when the mean Dice improves.

In [ ]:
for epoch in run.remaining_epochs():
    start = time.time()
    train_stats = training.train_one_epoch(model, train_loader, optimizer, scheduler, scaler, loss_fn, device, AMP,
                                           max_grad_norm=12.0)
    row = {**train_stats, 'lr': optimizer.param_groups[0]['lr']}
    mean_dice = None
    if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
        val_start = time.time()
        dice = segmentation.validation_dice(model, val_loader, device, ROI, SW_BATCH, SW_OVERLAP, AMP, THRESHOLD)
        mean_dice = float(np.mean(list(dice.values())))
        row.update({f'val_dice_{region}': value for region, value in dice.items()})
        row.update(val_mean_dice=mean_dice, val_seconds=time.time() - val_start)
    row['epoch_seconds'] = time.time() - start
    improved = run.end_epoch(epoch, row, metric=mean_dice)
    message = f'epoch {epoch:3d}/{EPOCHS} | loss {train_stats["train_loss"]:.4f} | {row["epoch_seconds"] / 60:.1f} min'
    if mean_dice is not None:
        message += (' | val Dice ' + ' '.join(f'{region} {dice[region]:.3f}' for region in config.REGIONS)
                    + f' mean {mean_dice:.3f}' + ('  * best' if improved else ''))
    print(message + f' | ~{run.hours_left(epoch):.1f} h left')
if not run.best_path.is_file():
    raise RuntimeError('No best.pt was saved - training did not validate even once.')
print(f'Training finished. Best validation mean Dice {run.best_metric:.4f} at epoch {run.best_epoch}.')

## 6 · Final evaluation on ALL validation and test patients, with localization
For every patient: predicted mask (NIfTI, BraTS labels), probabilities, Dice / IoU / HD95 / sensitivity / precision,
localization of the predicted and the true tumor, and the localization error. Each patient is saved as soon as it is done,
so this step also resumes after a disconnect.

In [ ]:
best = training.load_model_weights(run.best_path, device)
model.load_state_dict(best['model'])
model.eval()
print(f'Loaded best.pt from epoch {best["epoch"]} (val mean Dice {best["metric"]:.4f})')

results, voxel_confusion = {}, {}
for name in ('val', 'test'):
    out_dir = run_dir / 'predictions' / name
    evaluation.evaluate_segmentation_split(
        model, case_ids[name], cases_dir, out_dir, device, roi=ROI, sw_batch=SW_BATCH, overlap=SW_OVERLAP, amp=AMP,
        threshold=THRESHOLD, et_min_voxels=ET_MIN_VOXELS, save_probabilities=SAVE_PROBABILITIES,
        num_workers=NUM_WORKERS, split_name=name)
    results[name], voxel_confusion[name] = evaluation.collect_segmentation_split(out_dir, name)
    fileio.save_csv(results[name], run_dir / 'predictions' / f'{name}_metrics.csv')
    np.save(run_dir / 'predictions' / f'{name}_voxel_confusion.npy', voxel_confusion[name])

summary_by_split = {name: evaluation.summarize_segmentation(table) for name, table in results.items()}
overview = pd.DataFrame({name: {f'{metric} {region}': summary[f'{metric}_{region}_mean']
                                for metric in ('dice', 'iou', 'hd95') for region in config.REGIONS}
                         for name, summary in summary_by_split.items()})
display(overview.round(4))

In [ ]:
# Example localization reports (what the app will show), and inference speed / exports
for _, row in results['test'].head(5).iterrows():
    print(f'{row["case_id"]}\n   predicted: {row["report_pred"]}\n   expert   : {row["report_gt"]}\n'
          f'   centre error {row["centroid_error_mm_WT"]:.1f} mm, Dice WT {row["dice_WT"]:.3f}')
fileio.save_csv(results['test'][['case_id', 'report_pred', 'report_gt']],
                run_dir / 'predictions' / 'localization_reports_test.csv')

speed = {f'{device.type}_seconds_per_case_mean': float(results['test']['inference_seconds'].mean())}
try:
    example_case = dataset.load_case(cases_dir, case_ids['test'][0])
    cpu_model = copy.deepcopy(model).cpu().eval()
    start = time.time()
    segmentation.predict_volume(cpu_model, example_case.normalized(), torch.device('cpu'), ROI, 1, SW_OVERLAP, False)
    speed['cpu_seconds_one_case'] = time.time() - start
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        traced = torch.jit.trace(cpu_model, torch.zeros((1, 4) + tuple(ROI)))
    traced.save(str(run_dir / 'export' / 'segmenter_torchscript.pt'))
    print('TorchScript export saved.')
    del cpu_model
except Exception as error:      # the benchmark / export is a bonus; never fail the run because of it
    print('CPU benchmark / export skipped:', error)
print('Inference time:', {key: round(value, 2) for key, value in speed.items()})

summary = {'run_name': RUN_NAME, 'quick_run': QUICK_RUN, 'model_name': MODEL_NAME, 'parameters': n_params,
           'best_epoch': run.best_epoch, 'best_val_mean_dice_during_training': run.best_metric,
           'epochs_trained': len(run.history), 'training_hours': run.training_hours,
           'results': summary_by_split, 'inference': speed, 'threshold': THRESHOLD, 'et_min_voxels': ET_MIN_VOXELS}
fileio.save_json(summary, run_dir / 'summary.json')
# Everything the offline app needs to use this model:
fileio.save_json({'task': 'segmentation', 'weights': 'checkpoints/best.pt', 'model_name': MODEL_NAME, 'roi': list(ROI),
                  'in_channels': list(config.MODALITIES), 'out_channels': list(config.REGIONS), 'threshold': THRESHOLD,
                  'et_min_voxels': ET_MIN_VOXELS, 'sw_overlap': SW_OVERLAP,
                  'normalization': f'z-score per modality over non-zero voxels, clip +-{config.NORM_CLIP}',
                  'output_labels': 'BraTS: 1 necrotic core, 2 edema, 4 enhancing tumor'},
                 run_dir / 'model_card.json')
print('Saved summary.json and model_card.json')

## 7 · Graphs

In [ ]:
FIG = run_dir / 'figures'
history = pd.DataFrame(run.history)
fig = plots.training_curves(run.history, [
    ('Training loss (BCE + Dice)', [('train_loss', 'train')], 'loss'),
    ('Validation Dice per region', [(f'val_dice_{region}', region) for region in config.REGIONS], 'Dice'),
    ('Validation mean Dice', [('val_mean_dice', 'mean')], 'Dice'),
], title=f'Segmentation training - {MODEL_NAME}')
plots.save_fig(fig, FIG, '02_training_curves', data=history)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(history['epoch'], history['lr'], color=plots.PALETTE[0])
axes[0].set_title('Learning rate')
axes[1].bar(history['epoch'], history['epoch_seconds'] / 60, color=plots.PALETTE[1])
axes[1].set_title('Time per epoch')
axes[1].set_ylabel('minutes')
axes[2].bar(history['epoch'], history['gpu_mem_gb'], color=plots.PALETTE[2])
axes[2].set_title('Peak GPU memory')
axes[2].set_ylabel('GB')
for ax in axes:
    ax.set_xlabel('epoch')
fig.tight_layout()
plots.save_fig(fig, FIG, '03_lr_time_memory', data=history[['epoch', 'lr', 'epoch_seconds', 'gpu_mem_gb']])

In [ ]:
region_colors = [plots.REGION_COLORS[region] for region in config.REGIONS]
for name in ('val', 'test'):
    table = results[name]
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.9))
    plots.box_panel(axes[0], [table[f'dice_{r}'] for r in config.REGIONS], config.REGIONS, region_colors, 'Dice', 'Dice')
    plots.box_panel(axes[1], [table[f'iou_{r}'] for r in config.REGIONS], config.REGIONS, region_colors, 'IoU',
                    'IoU (Jaccard)')
    plots.box_panel(axes[2], [table[f'hd95_{r}'] for r in config.REGIONS], config.REGIONS, region_colors, 'mm',
                    'HD95 (lower is better)')
    plots.box_panel(axes[3], [table[f'sensitivity_{r}'] for r in config.REGIONS]
                    + [table[f'precision_{r}'] for r in config.REGIONS],
                    [f'{r}\nsens' for r in config.REGIONS] + [f'{r}\nprec' for r in config.REGIONS],
                    region_colors + region_colors, '', 'Sensitivity / precision')
    fig.suptitle(f'Segmentation scores per patient - {name} ({len(table)} patients)', y=1.04)
    fig.tight_layout()
    plots.save_fig(fig, FIG, f'04_scores_{name}', data=table)

test_results = results['test']
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, region in zip(axes, config.REGIONS):
    plots.hist_panel(ax, test_results[f'dice_{region}'], bins=20, color=plots.REGION_COLORS[region],
                     title=f'{config.REGION_NAMES[region]} ({region})', xlabel='Dice')
fig.suptitle('Distribution of Dice over test patients', y=1.04)
fig.tight_layout()
plots.save_fig(fig, FIG, '05_dice_histograms_test')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, region in zip(axes, config.REGIONS):
    true_volume = test_results[f'vol_gt_{region}_cm3'].astype(float)
    ax.scatter(true_volume.clip(lower=0.01), test_results[f'dice_{region}'], s=12, color=plots.REGION_COLORS[region],
               alpha=0.7, linewidths=0)
    plots.log_axis(ax, 'x')
    ax.set_xlabel('true volume (cm³, log)')
    ax.set_ylabel('Dice')
    ax.set_title(region)
fig.suptitle('Small tumors are harder: Dice vs true volume (test)', y=1.04)
fig.tight_layout()
plots.save_fig(fig, FIG, '06_dice_vs_volume_test')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for j, region in enumerate(config.REGIONS):
    plots.scatter_identity(axes[0, j], test_results[f'vol_gt_{region}_cm3'], test_results[f'vol_pred_{region}_cm3'],
                           plots.REGION_COLORS[region], 'true volume (cm³)', 'predicted volume (cm³)', f'{region} volume')
    plots.bland_altman(axes[1, j], test_results[f'vol_pred_{region}_cm3'], test_results[f'vol_gt_{region}_cm3'],
                       plots.REGION_COLORS[region], f'{region} Bland-Altman')
fig.suptitle('Volume measurement agreement (test)', y=1.01)
fig.tight_layout()
plots.save_fig(fig, FIG, '07_volume_agreement_test')

In [ ]:
# Localization quality
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
plots.hist_panel(axes[0], test_results['centroid_error_mm_WT'], bins=20, title='Tumor centre error (WT)', xlabel='mm')
plots.hist_panel(axes[1], test_results['bbox_iou_WT'], bins=20, color=plots.PALETTE[1],
                 title='Bounding-box overlap (WT)', xlabel='3D IoU')
size_error = pd.DataFrame({axis: test_results[f'pred_extent_{axis}_mm_WT'].astype(float)
                           - test_results[f'gt_extent_{axis}_mm_WT'].astype(float) for axis in 'xyz'})
plots.box_panel(axes[2], [size_error[axis] for axis in 'xyz'], ['left-right', 'front-back', 'up-down'],
                ylabel='predicted - true (mm)', title='Size error per direction (WT)')
fig.tight_layout()
plots.save_fig(fig, FIG, '08_localization_errors_test',
               data=test_results[['case_id', 'centroid_error_mm_WT', 'bbox_iou_WT']].join(size_error))

hemispheres = pd.crosstab(test_results['gt_side_WT'].fillna('none'), test_results['pred_side_WT'].fillna('none'))
side_labels = sorted(set(hemispheres.index) | set(hemispheres.columns))
hemispheres = hemispheres.reindex(index=side_labels, columns=side_labels, fill_value=0)
plots.save_fig(plots.confusion_matrix_fig(hemispheres.values, side_labels, 'Hemisphere: expert vs predicted (test)'),
               FIG, '09_hemisphere_agreement_test', data=hemispheres.reset_index())

label_names = [config.LABEL_SHORT[i] for i in range(4)]
for name in ('val', 'test'):
    plots.save_fig(plots.confusion_matrix_fig(voxel_confusion[name], label_names,
                                              f'Voxel labels inside the brain - {name}', normalize=True),
                   FIG, f'10_voxel_confusion_{name}',
                   data=pd.DataFrame(voxel_confusion[name], index=label_names, columns=label_names).reset_index())

In [ ]:
# Best, median and worst test patients: expert vs predicted
def predicted_labels(case_id: str, case) -> np.ndarray:
    """Predicted labels inside the crop, from the saved probabilities (or recomputed if they were not saved)."""
    probs_path = run_dir / 'predictions' / 'test' / 'probs' / f'{case_id}_probs.npz'
    if probs_path.is_file():
        probs = np.load(probs_path)['probs'].astype(np.float32)
    else:
        probs = segmentation.predict_volume(model, case.normalized(), device, ROI, SW_BATCH, SW_OVERLAP, AMP)
    return segmentation.probs_to_labels(probs, THRESHOLD, ET_MIN_VOXELS)


def show_cases(shown_ids: list[str], title: str, name: str) -> None:
    fig, axes = plt.subplots(len(shown_ids), 5, figsize=(13, 2.7 * len(shown_ids)), squeeze=False)
    dice_by_case = test_results.set_index('case_id')['dice_WT']
    for row, case_id in enumerate(shown_ids):
        case = dataset.load_case(cases_dir, case_id)
        image = case.normalized()
        truth, pred = case.labels, predicted_labels(case_id, case)
        k = int(np.argmax((truth > 0).sum((0, 1)))) if truth.any() else truth.shape[2] // 2
        flair = plots.view2d(image[0, :, :, k])
        first = row == 0
        plots.show_gray(axes[row, 0], flair, 'FLAIR' if first else None)
        plots.show_gray(axes[row, 1], plots.view2d(image[2, :, :, k]), 'T1ce' if first else None)
        plots.show_labels(axes[row, 2], flair, plots.view2d(truth[:, :, k]), 'Expert' if first else None)
        plots.show_labels(axes[row, 3], flair, plots.view2d(pred[:, :, k]), 'Model' if first else None)
        plots.show_gray(axes[row, 4], flair, 'Outlines: expert / model' if first else None)
        plots.show_contours(axes[row, 4], plots.view2d(truth[:, :, k] > 0), plots.PALETTE[0])
        plots.show_contours(axes[row, 4], plots.view2d(pred[:, :, k] > 0), plots.PALETTE[1])
        axes[row, 0].text(-0.08, 0.5, f'{case_id[-5:]}\nDice WT {dice_by_case[case_id]:.3f}',
                          transform=axes[row, 0].transAxes, rotation=90, va='center', ha='right', fontsize=8)
    plots.label_legend(fig, ncol=3)
    fig.suptitle(title + ' (outlines: blue = expert, orange = model)', y=1.0)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    plots.save_fig(fig, FIG, name)


ranked = test_results.sort_values('dice_WT')['case_id'].tolist()
middle = len(ranked) // 2
picks = {'best': ranked[-3:][::-1], 'median': ranked[middle - 1:middle + 2] if len(ranked) >= 3 else ranked,
         'worst': ranked[:3]}
for i, (kind, shown_ids) in enumerate(picks.items()):
    show_cases(shown_ids, f'{kind.capitalize()} test patients by WT Dice', f'{11 + i}_{kind}_cases')

In [ ]:
# Three views of one test patient with the predicted tumor and its measured centre
case_id = ranked[middle]
case = dataset.load_case(cases_dir, case_id)
image = case.normalized()
pred = predicted_labels(case_id, case)
report = test_results.set_index('case_id').loc[case_id, 'report_pred']
tumor = pred > 0
ci, cj, ck = (int(round(v)) for v in (np.argwhere(tumor).mean(0) if tumor.any() else np.array(tumor.shape) // 2))
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
plots.show_labels(axes[0], plots.view2d(image[0, :, :, ck]), plots.view2d(pred[:, :, ck]), 'Axial')
axes[0].plot(ci, cj, marker='+', color='white', markersize=14, mew=2)
plots.show_labels(axes[1], plots.view_vertical(image[0, :, cj, :]), plots.view_vertical(pred[:, cj, :]), 'Coronal')
axes[1].plot(ci, image.shape[3] - 1 - ck, marker='+', color='white', markersize=14, mew=2)
plots.show_labels(axes[2], plots.view_vertical(image[0, ci, :, :]), plots.view_vertical(pred[ci, :, :]), 'Sagittal')
axes[2].plot(cj, image.shape[3] - 1 - ck, marker='+', color='white', markersize=14, mew=2)
plots.label_legend(fig)
fig.suptitle(f'{case_id}: {report}', y=1.02, fontsize=9)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plots.save_fig(fig, FIG, '14_localization_example')

In [ ]:
# Compare all segmentation runs finished so far
comparison = evaluation.compare_segmentation_runs(paths.runs / 'segmentation', QUICK_RUN)
display(comparison)
fileio.save_csv(comparison, paths.runs / 'segmentation' / ('comparison_quick.csv' if QUICK_RUN else 'comparison.csv'))
plots.save_fig(plots.run_comparison_fig(comparison, ['dice_WT', 'dice_TC', 'dice_ET', 'hd95_WT', 'params_M',
                                                     'cpu_s_per_case'], title='Segmentation runs compared (test set)'),
               FIG, '15_run_comparison', data=comparison)

## Done
Saved in `runs/segmentation/<RUN_NAME>/`: `checkpoints/best.pt` (for the app), `last.pt`, per-epoch weights, `history.csv`,
`predictions/{val,test}/` (masks as NIfTI, probabilities, per-patient metrics and localization), `summary.json`,
`model_card.json`, `figures/`, `logs/`, `code_snapshot/`, `export/`.

In [ ]:
env.flush_drive()